# Quantum Phase Estimation

Estimate the eigenphase of a unitary by reading it off a counting register, using the inverse QFT.

**Objectives:**
- Build the QPE circuit: a counting register of $t$ qubits plus one eigenstate qubit
- Estimate the phase of the T gate, whose eigenvalue on $|1>$ is $e^{2\pi i (1/8)}$, so $\phi = 1/8 = 0.001_2$
- Recover the estimate exactly for $t=3$ and watch precision improve as $t$ grows
- Connect QPE to eigenvalue estimation, Shor's algorithm, and quantum chemistry

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.

<!-- browser-runnable -->

In [ ]:
# Setup: Run this cell first
# Requires: pip install -e '.[dev]' from the project root (see `make setup`)

from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt
from lib.utils.statevector import statevector

# Use local simulator by default (free, instant)
device = LocalSimulator()

## 1. The question QPE answers

Given a unitary $U$ and one of its eigenvectors $|u>$, we have

$$U\,|u> = e^{2\pi i \phi}\,|u>, \qquad 0 \le \phi < 1.$$

The eigenvalue lives entirely in the **phase** $\phi$. Measuring $|u>$ in any basis cannot reveal $\phi$ directly: a global phase on a single register is invisible. QPE's trick is **phase kickback** through a register of *counting* qubits, followed by an **inverse Quantum Fourier Transform** that turns the accumulated phase into a binary number we can simply read out.

Our test unitary is the T gate. On the computational basis,

$$T\,|0> = |0>, \qquad T\,|1> = e^{i\pi/4}\,|1> = e^{2\pi i (1/8)}\,|1>.$$

So $|1>$ is an eigenvector with phase $\phi = 1/8$. In binary that is exactly $0.001_2$ — three bits, no remainder — which is why $t=3$ counting qubits will give an *exact* answer. Let us confirm the T-gate phase before building anything.

In [ ]:
# T|1> = exp(i*pi/4)|1>. Read the (1,1) entry of the T matrix from its action on |1>.
# Build a 1-qubit circuit that prepares |1> then applies T, and inspect the state vector.
t_check = Circuit().x(0).t(0)
sv = statevector(t_check)  # statevector() returns the full complex numpy state vector

# Big-endian, 1 qubit: index 0 -> |0>, index 1 -> |1>. The amplitude on |1> carries the phase.
phase_factor = sv[1]
phi_from_gate = np.angle(phase_factor) / (2 * np.pi)
print("amplitude on |1>:", np.round(phase_factor, 6))
print("phi extracted from T gate:", phi_from_gate)

# ASSERT: the T gate's eigenphase on |1> is exactly 1/8 (exact, from the state vector).
assert np.isclose(phase_factor, np.exp(1j * np.pi / 4))
assert np.isclose(phi_from_gate, 0.125)
print("OK: T|1> = exp(2*pi*i*(1/8))|1>, so phi = 1/8 = 0.001 in binary.")

## 2. Controlled powers of U

QPE applies *controlled* powers of $U$, with counting qubit $k$ controlling $U^{2^{p}}$. A controlled-T fires only when both control and target are $|1>$, adding the phase $\pi/4$ to the $|11>$ component — which is **exactly** `cphaseshift(control, target, pi/4)`. Raising T to the power $2^p$ just multiplies the angle:

$$\text{controlled-}U^{2^p} \;=\; \texttt{cphaseshift}(\text{control},\ \text{eig},\ \tfrac{\pi}{4}\cdot 2^{p}).$$

**Layout convention (critical).** Counting qubits occupy indices $0\ldots t-1$, with **qubit 0 as the most-significant bit** of the estimate; the eigenstate qubit is index $t$. To make qubit 0 carry the most significant phase bit, counting qubit $k$ must control the *largest* power, so we assign it the angle

$$\frac{\pi}{4}\cdot 2^{\,t-1-k}.$$

This pairs with the inverse QFT below: `qft_circuit` (Hadamards + controlled phases, then swaps) is the *forward* QFT with qubit 0 as MSB, and `.adjoint()` reverses the gate order and conjugates every gate to give the exact inverse QFT in the same qubit ordering — so the controlled-power assignment and the inverse-QFT readout share one consistent MSB-first convention. Let us verify a single controlled-T does exactly what we claim.

In [ ]:
# A controlled-T adds phase pi/4 to the |11> component and nothing else.
# Big-endian 2 qubits: index 3 = |11> (control=qubit0=1, target=qubit1=1).
ctrl_t = Circuit().x(0).x(1).cphaseshift(0, 1, np.pi / 4)
sv2 = statevector(ctrl_t)
print("state vector (|00>,|01>,|10>,|11>):")
print(np.round(sv2, 6))

# ASSERT: only |11> is populated and it carries exactly exp(i*pi/4).
assert np.isclose(sv2[3], np.exp(1j * np.pi / 4))
assert np.allclose(sv2[[0, 1, 2]], 0)
print("OK: cphaseshift(c, t, pi/4) IS a controlled-T (phase pi/4 on |11>).")

## 3. Build the QPE circuit

The recipe:

1. **Prepare the eigenstate.** `x(t)` flips the eigenstate qubit to $|1>$ (the T eigenvector).
2. **Superpose the counting register.** Hadamard every counting qubit $0\ldots t-1$.
3. **Kick back the phase.** For each counting qubit $k$, apply `cphaseshift(k, t, (pi/4) * 2**(t-1-k))`. Qubit $k=0$ gets the largest power and becomes the MSB.
4. **Inverse QFT the counting register.** Build the forward QFT on qubits $0\ldots t-1$ and apply its `.adjoint()`.

We build the forward QFT exactly as the shared `lib.circuits.qft_circuit` does — Hadamards interleaved with controlled phase rotations, finished by the bit-reversal swaps — so the inverse-QFT ordering is guaranteed to match step 3.

In [ ]:
def forward_qft(n):
    """Forward QFT on qubits 0..n-1, MSB = qubit 0 (matches lib.circuits.qft_circuit)."""
    qft = Circuit()
    for i in range(n):
        qft.h(i)
        for j in range(i + 1, n):
            qft.cphaseshift(j, i, np.pi / (2 ** (j - i)))
    for i in range(n // 2):
        qft.swap(i, n - i - 1)
    return qft


def qpe_t_gate(t):
    """QPE for the T gate. Counting qubits 0..t-1 (qubit 0 = MSB), eigenstate qubit = t."""
    eig = t  # eigenstate qubit index
    circuit = Circuit()
    circuit.x(eig)  # prepare eigenstate |1>
    for k in range(t):
        circuit.h(k)  # superpose the counting register
    for k in range(t):
        # controlled-U^(2^p): counting qubit k controls the largest power so it is the MSB
        angle = (np.pi / 4) * (2 ** (t - 1 - k))
        circuit.cphaseshift(k, eig, angle)
    # inverse QFT on the counting register, same MSB-first ordering
    circuit.add_circuit(forward_qft(t).adjoint())
    return circuit


qpe3 = qpe_t_gate(3)
print(qpe3)
print("\nqubit_count:", qpe3.qubit_count, " depth:", qpe3.depth)

## 4. Read the estimate exactly ($t=3$)

qcsim is **big-endian** and **measures all qubits at once** — there is no partial measurement. A 4-qubit circuit (3 counting + 1 eigenstate) yields a 4-character bitstring; the counting register is `bitstring[:3]` (with qubit 0 leftmost = MSB) and the eigenstate qubit is `bitstring[3]`.

We convert the counting substring to an integer, MSB-first, and divide by $2^t$:

$$\hat\phi = \frac{\texttt{int(bitstring[:t], 2)}}{2^{t}}.$$

Because $\phi = 1/8 = 0.001_2$ fits in $t=3$ bits, QPE concentrates **all** the probability on a single basis state. We read it off the exact state vector (deterministic) rather than by sampling.

In [ ]:
t = 3
circuit = qpe_t_gate(t)
n = circuit.qubit_count  # = t + 1

probs = np.abs(statevector(circuit)) ** 2
idx = int(np.argmax(probs))                 # highest-probability full basis state
bitstring = format(idx, "0" + str(n) + "b")  # big-endian: qubit 0 is leftmost

count_bits = bitstring[:t]                   # counting register (qubit 0 = MSB)
eig_bit = bitstring[t]                        # eigenstate qubit
value = int(count_bits, 2)
phi_hat = value / 2 ** t

print("most-probable bitstring:", bitstring, " probability:", round(probs[idx], 6))
print("counting register:", count_bits, "-> value", value, "-> phi_hat", phi_hat)
print("eigenstate qubit:", eig_bit, "(stays |1>, the T eigenvector)")

# ASSERT (exact, from the state vector): for t=3 the dominant basis state reads phi_hat == 0.125.
assert np.isclose(probs[idx], 1.0)          # all probability on one state -> exact
assert phi_hat == 0.125                       # 001 binary = 1/8
assert eig_bit == "1"                          # eigenstate undisturbed (phase kicked to counting reg)
print("OK: QPE recovers phi = 1/8 exactly with t=3 counting qubits.")

## 5. Now measure it (sampling demonstration)

The state vector is exact, but on real hardware we only get samples. Because $t=3$ resolves $\phi=1/8$ perfectly, every shot returns the same outcome. We run the simulator, slice the counting substring from each measured bitstring, and histogram the recovered $\hat\phi$ values. (Sampling is for illustration only — the tight asserts above use the exact state vector.)

In [ ]:
result = device.run(circuit, shots=2000).result()
counts = result.measurement_counts  # Counter of full bitstrings -> count
print("raw counts:", dict(counts))

# Aggregate by the counting-register substring and convert to phi_hat estimates.
phi_counts = {}
for bits, c in counts.items():
    v = int(bits[:t], 2)
    phi_counts[v / 2 ** t] = phi_counts.get(v / 2 ** t, 0) + c

phis = sorted(phi_counts)
heights = [phi_counts[p] for p in phis]
plt.figure(figsize=(7, 4))
plt.bar([str(p) for p in phis], heights, color="#4c72b0")
plt.axvline(0, color="#c44e52", linestyle="--", label="true phi = 1/8 = 0.125")
plt.xlabel("estimated phi_hat")
plt.ylabel("shots")
plt.title("QPE of the T gate (t=3): all shots land on 1/8")
plt.legend()
plt.tight_layout()
plt.show()

# The single populated outcome must decode to 1/8.
assert set(phi_counts) == {0.125}
print("OK: every shot decodes to phi_hat = 0.125.")

## 6. Precision improves with $t$

For the T gate, $\phi=1/8$ is already exact at $t=3$, so adding counting qubits keeps the answer at $1/8$ — the binary expansion simply gains trailing zeros (`001`, `0010`, `00100`, all equal to $1/8$). More counting qubits never *hurt* an exact phase; they buy resolution for phases that are **not** dyadic.

We confirm exactness at $t=3,4,5$ from the state vector, then make the resolution story concrete with a deliberately awkward phase $\phi = 0.1$ (not a finite binary fraction), where the best $t$-bit estimate steadily tightens toward the truth.

In [ ]:
# Exactness of the T-gate phase across t: phi_hat stays 1/8 and stays exact.
for tt in [3, 4, 5]:
    circ = qpe_t_gate(tt)
    nn = circ.qubit_count
    p = np.abs(statevector(circ)) ** 2
    j = int(np.argmax(p))
    b = format(j, "0" + str(nn) + "b")
    val = int(b[:tt], 2)
    est = val / 2 ** tt
    print(f"t={tt}: bits={b}  counting={b[:tt]}  phi_hat={est}  prob={p[j]:.4f}")
    # ASSERT: dyadic phase 1/8 is recovered exactly with probability 1 for every t >= 3.
    assert np.isclose(p[j], 1.0)
    assert est == 0.125
print("OK: a dyadic phase (1/8) is exact for all t >= 3.")

In [ ]:
# A non-dyadic phase shows precision genuinely improving with t.
# Use a general phase unitary diag(1, exp(2*pi*i*phi)) on |1> via cphaseshift powers.
def qpe_phase(t, phi):
    eig = t
    circ = Circuit().x(eig)
    for k in range(t):
        circ.h(k)
    for k in range(t):
        circ.cphaseshift(k, eig, 2 * np.pi * phi * (2 ** (t - 1 - k)))
    circ.add_circuit(forward_qft(t).adjoint())
    return circ

phi_true = 0.1  # = 0.000110011..._2, never terminates -> QPE can only approximate
errors = []
ts = [3, 4, 5]
for tt in ts:
    circ = qpe_phase(tt, phi_true)
    nn = circ.qubit_count
    p = np.abs(statevector(circ)) ** 2
    j = int(np.argmax(p))
    b = format(j, "0" + str(nn) + "b")
    est = int(b[:tt], 2) / 2 ** tt
    err = abs(est - phi_true)
    errors.append(err)
    print(f"t={tt}: best phi_hat={est:.5f}  |error|={err:.5f}")

# ASSERT: the best-estimate error is bounded by the resolution 2^-t at each t (exact, from state vector).
for tt, err in zip(ts, errors):
    assert err <= 2 ** (-tt) + 1e-9
print("OK: each t-bit best estimate lands within one resolution step (2^-t) of the true phase.")

## 7. Why this matters

QPE is **eigenvalue estimation** dressed as a circuit: feed it an eigenvector and it writes the eigenphase into a register you can read. That single primitive is the engine behind two flagship applications.

- **Shor's algorithm.** Factoring reduces to finding the period of modular exponentiation, which is the eigenphase of the "multiply by $a$ mod $N$" unitary. QPE reads that phase; continued fractions turn it into the period.
- **Quantum chemistry.** The ground-state energy of a molecule is the lowest eigenvalue of its Hamiltonian. Encode time evolution $U = e^{-iHt}$, prepare an approximate ground state, and QPE estimates the energy to chemical accuracy. (This is the eigenvalue half of the story you will meet again in `05-quantum-chemistry`.)

The inverse QFT you applied here is the same transform you built in the previous notebook — pointed at a different question. QFT *creates* periodic structure; QPE *reads* a phase out of it.

## Exercises

Try these to deepen your understanding. Scaffolds only — no solutions.

1. Estimate the phase of the **S gate** ($S|1> = e^{i\pi/2}|1>$, so $\phi = 1/4 = 0.01_2$). How many counting qubits do you need for an exact answer?
2. Use the **wrong eigenstate**: prepare $|0>$ instead of $|1>$ (drop the `x(eig)`). What phase does QPE report, and why? (Hint: $T|0> = |0>$.)
3. Sweep $t$ from 2 to 6 for the non-dyadic phase $\phi = 0.1$ and plot the best-estimate error versus $t$ on a log scale.

In [ ]:
# Exercise 1 scaffold: QPE for the S gate (phi = 1/4).
# S|1> = exp(i*pi/2)|1>, so a controlled-S is cphaseshift(control, eig, pi/2).
#
# def qpe_s_gate(t):
#     eig = t
#     circ = Circuit().x(eig)
#     for k in range(t):
#         circ.h(k)
#     for k in range(t):
#         circ.cphaseshift(k, eig, (np.pi / 2) * (2 ** (t - 1 - k)))  # angle pi/2 for S
#     circ.add_circuit(forward_qft(t).adjoint())
#     return circ
#
# circ = qpe_s_gate(2)
# probs = np.abs(statevector(circ)) ** 2
# idx = int(np.argmax(probs))
# bits = format(idx, "0" + str(circ.qubit_count) + "b")
# print("phi_hat =", int(bits[:2], 2) / 2 ** 2, "(expect 0.25)")

In [ ]:
# Exercise 3 scaffold: error vs. t for a non-dyadic phase.
#
# phi_true = 0.1
# ts = range(2, 7)
# errs = []
# for tt in ts:
#     circ = qpe_phase(tt, phi_true)
#     probs = np.abs(statevector(circ)) ** 2
#     j = int(np.argmax(probs))
#     b = format(j, "0" + str(circ.qubit_count) + "b")
#     errs.append(abs(int(b[:tt], 2) / 2 ** tt - phi_true))
# plt.semilogy(list(ts), errs, marker="o")
# plt.xlabel("t (counting qubits)"); plt.ylabel("|error|"); plt.show()

## Summary

- **QPE reads an eigenphase** $\phi$ (where $U|u> = e^{2\pi i\phi}|u>$) into a counting register via phase kickback and an inverse QFT.
- **Layout:** counting qubits $0\ldots t-1$ with qubit 0 as MSB, eigenstate qubit at index $t$; prepare $|u>$, Hadamard the counting register, apply controlled-$U^{2^p}$, then the inverse QFT.
- For the T gate, $\phi = 1/8 = 0.001_2$ is **exact at $t=3$** — all probability lands on one bitstring, which decodes to $0.125$.
- **More counting qubits buy resolution:** dyadic phases stay exact; non-dyadic phases like $0.1$ approach the truth within $2^{-t}$ as $t$ grows.
- QPE is the **eigenvalue-estimation primitive** behind Shor's factoring and quantum-chemistry ground-state energy.

**Next:** [`05-qaoa-maxcut.ipynb`](05-qaoa-maxcut.ipynb) — trade the exact, fault-tolerant phase readout for a near-term *variational* approach to combinatorial optimization.